In [7]:
INPUT_FILE = "puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_9223372036854775807.xlsx"

In [1]:
import os
import re
from datetime import datetime, date, time
from dateutil.parser import parse
from openpyxl import load_workbook

# === НАСТРОЙКИ ===
INPUT_DIR = "puid2024"  # <-- папка с входными XLSX-файлами
HEADER_XLSX = "header.xlsx"  # <-- шаблон/хедер
OUTPUT_BASE = "output_data/"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# === РЕГУЛЯРКИ ===
km_re = re.compile(r"км \d+\+\d+", re.IGNORECASE)

# === ФУНКЦИИ ===


def safe_filename(s: str) -> str:
    s = s.strip()
    s = s.replace('"', "")
    s = s.replace(" ", "_")
    s = s.replace("/", "_")
    s = re.sub(r'[\\:*?"<>|]', "", s)
    s = re.sub(r"_+", "_", s)
    return s


def is_datetime(value) -> bool:
    if isinstance(value, (datetime, date, time)):
        return True
    try:
        parse(str(value), fuzzy=False)
        return True
    except (ValueError, TypeError):
        return False


def is_valid_highway_line(s) -> bool:
    s_str = (str(s) if s is not None else "").strip()
    if not s_str or s_str.lower() in ["итого", "среднее", "%"]:
        return False
    if km_re.search(s_str):
        return False
    if is_datetime(s):
        return False
    return True


def process_xlsx(input_xlsx_path: str) -> None:
    """Режем первый лист входного XLSX на блоки и сохраняем каждый блок в .xlsx с header сверху."""
    # Локальные состояния для конкретного входного файла
    current_highway = None
    current_km = None
    out_wb = None  # активная выходная книга (копия HEADER_XLSX)
    out_ws = None  # лист, куда пишем
    out_path = None  # путь к текущему выходному файлу
    prev_row = None

    # читаем первый лист входного файла (значения формул)
    src_wb = load_workbook(input_xlsx_path, data_only=True, read_only=True)
    try:
        src_ws = src_wb.worksheets[0]

        for row in src_ws.iter_rows(values_only=True):
            if not row or (row[0] is None) or (str(row[0]).strip() == ""):
                prev_row = row
                continue

            first_cell_raw = row[0]
            first_cell = str(first_cell_raw).strip()

            # Обнаружение новой КМО
            if km_re.search(first_cell):
                # Закрыть предыдущий выходной файл, если был
                if out_wb and out_path:
                    out_wb.save(out_path)
                    out_wb.close()
                    out_wb, out_ws, out_path = None, None, None

                current_km = safe_filename(first_cell)

                # Попытка взять трассу из предыдущей строки
                prev_val = prev_row[0] if prev_row and len(prev_row) > 0 else ""
                if is_valid_highway_line(prev_val):
                    current_highway = safe_filename(str(prev_val))

                if not current_highway:
                    print(f"[!] Пропущен блок КМО: {current_km} (трасса не определена)")
                    prev_row = row
                    continue

                # Создаём выходной XLSX из шаблона header.xlsx
                highway_dir = os.path.join(OUTPUT_BASE, current_highway)
                os.makedirs(highway_dir, exist_ok=True)
                out_path = os.path.join(highway_dir, f"{current_km}.xlsx")

                # Загружаем шаблон без read_only, чтобы дописывать
                out_wb = load_workbook(HEADER_XLSX)
                out_ws = out_wb.active  # дописываем данные блока в конец этого листа
                prev_row = row
                continue

            # Строка данных блока
            if out_ws and is_datetime(first_cell_raw):
                out_ws.append(list(row))

            prev_row = row

        # Сохранить последний открытый файл
        if out_wb and out_path:
            out_wb.save(out_path)
            out_wb.close()

    finally:
        # Закрыть исходную книгу
        try:
            src_wb.close()
        except Exception:
            pass


# === ОБРАБОТКА ВСЕХ ФАЙЛОВ В ПАПКЕ ===
for fname in os.listdir(INPUT_DIR):
    if (
        not fname.lower().endswith(".xlsx")
        or fname.startswith("~$")
        or fname == os.path.basename(HEADER_XLSX)
    ):
        continue

    in_path = os.path.join(INPUT_DIR, fname)
    print(f"▶ Обработка: {in_path}")
    try:
        process_xlsx(in_path)
        print(f"✅ Готово: {in_path}")
    except Exception as e:
        print(f"❌ Ошибка при обработке {in_path}: {e}")

▶ Обработка: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_0(1).xlsx
✅ Готово: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_0(1).xlsx
▶ Обработка: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_0(2).xlsx
✅ Готово: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_0(2).xlsx
▶ Обработка: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_0.xlsx
✅ Готово: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_0.xlsx
▶ Обработка: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_1754560022.xlsx
✅ Готово: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_1754560022.xlsx
▶ Обработка: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_1754560201.xlsx
✅ Готово: puid2024\Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2024_31.12.2024_1754560201.xl

In [9]:
import os

OUTPUT_BASE = "output_data/"

highway_km_dict = {}

for root, dirs, files in os.walk(OUTPUT_BASE):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        km_files = []

        for file in os.listdir(dir_path):
            if file.endswith(".csv"):
                km_name = os.path.splitext(file)[0]  # убрать .csv
                km_files.append(km_name)

        highway_km_dict[dir_name] = sorted(km_files)

# Пример вывода
for highway, kms in highway_km_dict.items():
    print(f"{highway}: {len(kms)} км-точек")
    for km in kms:
        print(f"  - {km}")

А-289_Краснодар_-_Славянск-на-Кубани_-_Темрюк: 3 км-точек
  - км_37+338_а_д_А-289_Краснодар_-_Темрюк
  - км_69+370_а_д_А-289_Краснодар_-_Темрюк
  - км_7+350_а_д_А-289_Краснодар_-_Темрюк
А-146_Краснодар_-_Новороссийск_-_Верхнебаканский: 3 км-точек
  - км_130+810_а_д_А-146_Краснодар-Верхнебаканский
  - км_32+000_а_д_А-146_Краснодар-Верхнебаканский
  - км_53+800_а_д_А-146_Краснодар-Верхнебаканский
А-160_Майкоп_-_Усть-Лабинск_-_Кореновск: 1 км-точек
  - км_58+990_а_д_А-160_Майкоп-Кореновск
А-290_Новороссийск_-_Керченский_пролив_-_Керчь: 2 км-точек
  - км_10+200_а_д_А-290_Новороссийск-Керчь
  - км_28+500_а_д_А-290_Новороссийск-Керчь
03_ОП_РЗ_03К-001_г.Краснодар_-_г.Ейск: 1 км-точек
  - км_47+243_а_д_г.Краснодар_-_г.Ейск
М-4_Дон_Москва_–_Воронеж_–_Ростов-на-Дону_–_Краснодар_–_Новороссийск: 5 км-точек
  - км_1278+575_а_д_М-4_Дон_Москва_–_Новороссийск
  - км_1301+500_а_д_М-4_Дон_Москва_–_Новороссийск
  - км_1389+000_а_д_М-4_Дон_Москва_–_Новороссийск
  - км_1430+150_а_д_М-4_Дон_Москва_–_Новорос

In [8]:
highway_km_dict

{'А-289_Краснодар_-_Славянск-на-Кубани_-_Темрюк': ['км_37+338_а_д_А-289_Краснодар_-_Темрюк',
  'км_69+370_а_д_А-289_Краснодар_-_Темрюк',
  'км_7+350_а_д_А-289_Краснодар_-_Темрюк'],
 'А-146_Краснодар_-_Новороссийск_-_Верхнебаканский': ['км_130+810_а_д_А-146_Краснодар-Верхнебаканский',
  'км_32+000_а_д_А-146_Краснодар-Верхнебаканский',
  'км_53+800_а_д_А-146_Краснодар-Верхнебаканский'],
 'А-160_Майкоп_-_Усть-Лабинск_-_Кореновск': ['км_58+990_а_д_А-160_Майкоп-Кореновск'],
 'А-290_Новороссийск_-_Керченский_пролив_-_Керчь': ['км_10+200_а_д_А-290_Новороссийск-Керчь',
  'км_28+500_а_д_А-290_Новороссийск-Керчь'],
 '03_ОП_РЗ_03К-001_г.Краснодар_-_г.Ейск': ['км_47+243_а_д_г.Краснодар_-_г.Ейск'],
 'М-4_Дон_Москва_–_Воронеж_–_Ростов-на-Дону_–_Краснодар_–_Новороссийск': ['км_1278+575_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1301+500_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1389+000_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1430+150_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1443+500_а_д_М-4_Дон_Мо